# Building agents with the Claude Agent SDK

One runnable example per module. Every cell ends in an `assert`, so a broken example fails the
run instead of printing something plausible.

| Module | Example |
|---|---|
| 1 | The loop, branching on `stop_reason` — and the three stopping rules that get it wrong |
| 2 | The same tools as an in-process MCP server; `tool_choice` auto / any / forced |
| 3 | A PreToolUse gate and ceiling that always run; a PostToolUse hook that normalises dates |
| 4 | Structured errors: retry a timeout, never retry a policy violation; empty is not failure |
| 5 | Fixed steps (prompt chaining) and a multi-part request split three ways |
| 6 | A coordinator with search, analysis and writing subagents, spawned in parallel |
| 7 | A scratchpad file, then `resume` and `fork_session` |
| 8 | An escalation handoff a human can act on without reading the conversation |
| — | Capstone: all of it on one message with four problems |

## Setup

In [1]:
%pip install -q claude-agent-sdk anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /Users/akshika47/Documents/GitHub/AI-Internship/claude-architect-bootcamp/week-1/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio
import json
import os
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

for candidate in [Path(".env"), Path("../week-2/.env"), Path("../week-1/.env"),
                  Path("../week-1/demo-ui/.env.local")]:
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), "Put ANTHROPIC_API_KEY in .env"
MODEL = "claude-sonnet-4-6"
print("ready")

ready


Three small dicts stand in for the backend. Every example here is about control flow, so the data stays deliberately simple.

In [3]:
TODAY = "2026-10-16"

CUSTOMERS = {
    "alice@example.com": {"customer_id": "CUST-001", "name": "Alice Chen", "tier": "Gold"},
}

ORDERS = {
    "ORD-123": {"customer_id": "CUST-001", "amount": 149.99, "status": "delivered",
                "delivered_on": "2026-09-28", "condition": "damaged", "charges": 1},
    "ORD-456": {"customer_id": "CUST-001", "amount": 89.00, "status": "delivered",
                "delivered_on": "2026-09-22", "condition": "ok", "charges": 2},
    "ORD-789": {"customer_id": "CUST-001", "amount": 812.40, "status": "delivered",
                "delivered_on": "2026-09-06", "condition": "ok", "charges": 1},
    # Inside the 30-day window and genuinely damaged, so the only thing that can
    # stop a refund here is the $500 ceiling in Module 3.
    "ORD-999": {"customer_id": "CUST-001", "amount": 720.00, "status": "delivered",
                "delivered_on": "2026-10-10", "condition": "damaged", "charges": 1},
}

POLICY = {
    "2.2": "Returns are accepted within 30 days of delivery. Day 31 onward is refused.",
    "3.5": "Refunds above $500 require a human agent. The automated system must escalate.",
    "4.1": "A duplicate charge is refunded in full without a return.",
}

REFUND_LOG: list[dict] = []

ALICE_TICKET = (
    "I'm alice@example.com. "
    "ORD-123 arrived damaged and I want a refund. "
    "ORD-456 got charged to my card twice. "
    "I bought shoes on ORD-789 forty days ago. Can I still return them? "
    "If this isn't sorted today I'm filing a chargeback."
)
print(ALICE_TICKET)

I'm alice@example.com. ORD-123 arrived damaged and I want a refund. ORD-456 got charged to my card twice. I bought shoes on ORD-789 forty days ago. Can I still return them? If this isn't sorted today I'm filing a chargeback.


## Module 1 — The agentic loop

An agent is a loop. You send a message. Claude answers or asks for a tool. You run the tool, add the
result to the history, and ask again. `stop_reason` tells you which one happened.

In [4]:
from anthropic import Anthropic

client = Anthropic()

RAW_TOOLS = [
    {"name": "get_customer",
     "description": "Look up a customer by email. Returns customer_id, name, tier.",
     "input_schema": {"type": "object", "properties": {"email": {"type": "string"}},
                      "required": ["email"]}},
    {"name": "lookup_order",
     "description": "Look up an order by ID. Returns status, delivered_on, amount, condition, charges.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
]

RAW_IMPLS = {
    "get_customer": lambda a: CUSTOMERS.get(a["email"], {"error": "unknown customer"}),
    "lookup_order": lambda a: ORDERS.get(a["order_id"], {"error": "unknown order"}),
}


def run_loop(user_message: str, max_turns: int = 8):
    """The loop. It stops on stop_reason, never on what the text says."""
    messages = [{"role": "user", "content": user_message}]
    transcript = []
    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=1024, tools=RAW_TOOLS, messages=messages,
            system="You are a Northwind support agent. Verify the customer, then answer.",
        )
        text = "".join(b.text for b in resp.content if b.type == "text")
        calls = [b for b in resp.content if b.type == "tool_use"]
        transcript.append({"turn": turn, "stop_reason": resp.stop_reason,
                           "text": text, "tools": [c.name for c in calls]})
        print(f"turn {turn}  stop_reason={resp.stop_reason:10} tools={[c.name for c in calls]}")

        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return text, transcript

        # Every tool result goes back into the history, or Claude cannot see what happened.
        results = [{"type": "tool_result", "tool_use_id": c.id,
                    "content": json.dumps(RAW_IMPLS[c.name](c.input))} for c in calls]
        messages.append({"role": "user", "content": results})
    return "", transcript


answer, TRANSCRIPT = run_loop(
    "I'm alice@example.com. What is the condition of ORD-123, and am I a Gold member?")
print("\nanswer:", answer)

assert TRANSCRIPT[-1]["stop_reason"] == "end_turn", "the loop must end on end_turn"
assert any(t["tools"] for t in TRANSCRIPT), "the agent should have used at least one tool"

turn 0  stop_reason=tool_use   tools=['get_customer', 'lookup_order']


turn 1  stop_reason=end_turn   tools=[]

answer: Here's what I found, Alice:

1. **Gold Member Status:** Yes, you are indeed a **Gold** member! 🎉

2. **Order ORD-123 Condition:** Unfortunately, the condition of your order is listed as **damaged**. I'm sorry to hear that! Since the order was delivered on **September 28, 2026**, you may want to file a claim or request a replacement. Would you like help with that?


### The three stopping rules that get it wrong

No new API calls — the recorded transcript above is replayed through each rule. A rule is wrong when
it stops on a turn whose `stop_reason` was still `tool_use`: work was outstanding.

In [5]:
def stops_at(transcript, rule):
    for t in transcript:
        if rule(t):
            return t
    return transcript[-1]


RULES = {
    "text says done":   lambda t: any(w in t["text"].lower() for w in ("done", "finished")),
    "any text at all":  lambda t: bool(t["text"].strip()),
    "stop_reason":      lambda t: t["stop_reason"] != "tool_use",
}

for name, rule in RULES.items():
    stop = stops_at(TRANSCRIPT, rule)
    unfinished = stop["stop_reason"] == "tool_use"
    print(f"{name:>16}: stops on turn {stop['turn']} "
          f"(stop_reason={stop['stop_reason']}) "
          f"{'-- WORK STILL OUTSTANDING' if unfinished else '-- complete'}")

assert stops_at(TRANSCRIPT, RULES["stop_reason"])["stop_reason"] != "tool_use"
assert stops_at(TRANSCRIPT, RULES["any text at all"])["turn"] <= TRANSCRIPT[-1]["turn"]

  text says done: stops on turn 1 (stop_reason=end_turn) -- complete
 any text at all: stops on turn 0 (stop_reason=tool_use) -- WORK STILL OUTSTANDING
     stop_reason: stops on turn 1 (stop_reason=end_turn) -- complete


## Module 2 — Giving the agent tools

The same tools become an in-process MCP server, and `query()` owns the loop. Note how few tools each
agent gets: four or five pick better than eighteen.

In [6]:
from claude_agent_sdk import (
    AgentDefinition, AssistantMessage, ClaudeAgentOptions, ClaudeSDKClient, HookMatcher,
    ResultMessage, TextBlock, ToolUseBlock, create_sdk_mcp_server, query, tool,
)


def ok(payload: dict) -> dict:
    return {"content": [{"type": "text", "text": json.dumps(payload)}]}


def err(category: str, retryable: bool, description: str, **extra) -> dict:
    """Module 4's error shape, defined once and used by every tool below."""
    payload = {"ok": False, "category": category, "retryable": retryable,
               "description": description, **extra}
    return {"content": [{"type": "text", "text": json.dumps(payload)}], "is_error": True}


@tool("get_customer", "Look up one customer by email. Returns customer_id, name and tier. "
      "Use to verify identity before any order operation. Read only.", {"email": str})
async def t_get_customer(args):
    c = CUSTOMERS.get(args["email"])
    return ok(c) if c else err("validation", False, "No customer with that email.")


@tool("lookup_order", "Look up one order by ID. Returns status, delivered_on, amount, "
      "condition and how many times it was charged. Read only.", {"order_id": str})
async def t_lookup_order(args):
    o = ORDERS.get(args["order_id"])
    return ok({"order_id": args["order_id"], **o}) if o else err(
        "validation", False, "Unknown order id.")


@tool("read_policy", "Read one clause of the support policy by section, e.g. '2.2'. Read only.",
      {"section": str})
async def t_read_policy(args):
    text = POLICY.get(args["section"])
    return ok({"section": args["section"], "text": text}) if text else err(
        "validation", False, f"No policy section {args['section']}.")


@tool("process_refund", "Refund an amount against one order. Mutating. Only after the order is "
      "looked up and the customer verified.", {"order_id": str, "amount": float})
async def t_process_refund(args):
    REFUND_LOG.append({"order_id": args["order_id"], "amount": args["amount"]})
    return ok({"refunded": args["amount"], "order_id": args["order_id"]})


@tool("escalate_to_human", "Hand the case to a human. Supply customer_id, root_cause, amount and "
      "recommended_action: the human cannot see this conversation.",
      {"customer_id": str, "root_cause": str, "amount": float, "recommended_action": str})
async def t_escalate(args):
    HANDOFFS.append(args)
    return ok({"ticket": "ESC-9001", **args})


HANDOFFS: list[dict] = []

northwind = create_sdk_mcp_server(
    name="northwind", version="1.0.0",
    tools=[t_get_customer, t_lookup_order, t_read_policy, t_process_refund, t_escalate],
)
print([f"mcp__northwind__{n}" for n in
       ("get_customer", "lookup_order", "read_policy", "process_refund", "escalate_to_human")])

['mcp__northwind__get_customer', 'mcp__northwind__lookup_order', 'mcp__northwind__read_policy', 'mcp__northwind__process_refund', 'mcp__northwind__escalate_to_human']


`setting_sources=[]` stops the SDK reading `CLAUDE.md`, skills and hooks from your machine, so this notebook runs the same everywhere.

In [7]:
async def run(prompt: str, *, client_mode: bool = False, **overrides):
    """Run one agent and print every tool call. Returns (final_text, tool_names)."""
    opts = dict(
        mcp_servers={"northwind": northwind},
        allowed_tools=["Agent", "Task", "mcp__northwind__*"],
        setting_sources=[],
        permission_mode="bypassPermissions",   # no interactive prompt in a notebook
        max_turns=10,
    )
    opts.update(overrides)
    options = ClaudeAgentOptions(**opts)
    used, result_text, session_id = [], "", None

    def handle(message):
        nonlocal result_text, session_id
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    used.append(block.name)
                    if block.name in ("Agent", "Task"):
                        print("spawned:", block.input.get("subagent_type"),
                              "|", str(block.input.get("prompt", ""))[:70])
                    else:
                        print("   tool:", block.name, json.dumps(block.input)[:90])
        if isinstance(message, ResultMessage):
            session_id = message.session_id
            result_text = (message.result or "") if message.subtype == "success" else result_text
            if message.subtype != "success":
                print("(ended:", message.subtype, ")")

    if client_mode:
        async with ClaudeSDKClient(options=options) as c:
            await c.query(prompt)
            async for message in c.receive_response():
                handle(message)
    else:
        async for message in query(prompt=prompt, options=options):
            handle(message)

    print("\nRESULT:", result_text[:400])
    return result_text, used, session_id


text, used, _ = await run(
    "Customer alice@example.com asks whether ORD-789 is still returnable. "
    f"Today is {TODAY}. Check the policy before you answer.")

assert "mcp__northwind__lookup_order" in used, f"expected an order lookup, got {used}"
assert text, "the agent returned no answer"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__northwind__read_policy,mcp__northwind__lookup_order,mcp__northwind_


   tool: mcp__northwind__get_customer {"email": "alice@example.com"}


   tool: mcp__northwind__lookup_order {"order_id": "ORD-789"}


   tool: mcp__northwind__read_policy {"section": "2.1"}


...

   tool: mcp__northwind__read_policy {"section": "1.1"}


   tool: mcp__northwind__read_policy {"section": "2"}


   tool: mcp__northwind__read_policy {"section": "2.4"}


   tool: mcp__northwind__read_policy {"section": "4.1"}



RESULT: **No — ORD-789 is outside the return window.**

**The check:**

| | |
|---|---|
| Customer | Alice Chen, CUST-001, Gold — matches the order |
| Delivered | 2026-09-06 |
| Today | 2026-10-16 → **day 40** |
| Policy §2.2 | "Returns are accepted within 30 days of delivery. Day 31 onward is refused." |
| Window closed | 2026-10-06, ten days ago |

**Things I checked so the "no" isn't premature:**

- *


### `tool_choice`: auto, any, forced

Same prompt, three settings, on the raw Messages API so the choice is visible in one response.

In [8]:
PROMPT = "Hello. Nothing is wrong with my order, I just wanted to say thanks."

choices = {
    "auto":   {"type": "auto"},
    "any":    {"type": "any"},
    "forced": {"type": "tool", "name": "get_customer"},
}

observed = {}
for label, tool_choice in choices.items():
    resp = client.messages.create(
        model=MODEL, max_tokens=512, tools=RAW_TOOLS, tool_choice=tool_choice,
        messages=[{"role": "user", "content": PROMPT}])
    names = [b.name for b in resp.content if b.type == "tool_use"]
    observed[label] = names
    print(f"{label:>7}: stop_reason={resp.stop_reason:10} tools={names}")

assert observed["forced"] == ["get_customer"], "forced must call the tool you named"
assert observed["any"], "any must call some tool"

   auto: stop_reason=end_turn   tools=[]


    any: stop_reason=tool_use   tools=['get_customer']


 forced: stop_reason=tool_use   tools=['get_customer']


## Module 3 — Guardrails with hooks

A prompt works most of the time. For money and identity, most of the time is not good enough. Hooks
are code, and code always runs.

Two rules below, neither of them in the prompt:

1. **Prerequisite gate** — `process_refund` is blocked until `get_customer` has returned a verified ID.
2. **Ceiling** — any refund over $500 is denied and sent to a human.

In [9]:
REFUND_CEILING = 500.00
VERIFIED: set[str] = set()
DENIALS: list[str] = []


async def record_verification(input_data, tool_use_id, context):
    """PostToolUse on get_customer: remember that identity was established."""
    payload = input_data.get("tool_response")
    text = json.dumps(payload)
    for cid in ("CUST-001",):
        if cid in text:
            VERIFIED.add(cid)
    return {}


async def gate_and_ceiling(input_data, tool_use_id, context):
    """PreToolUse on process_refund. Runs before the tool, in any agent."""
    amount = float(input_data["tool_input"].get("amount", 0))
    if not VERIFIED:
        reason = "No verified customer yet. Call get_customer first, then retry the refund."
    elif amount > REFUND_CEILING:
        reason = (f"${amount:.2f} is over the ${REFUND_CEILING:.2f} automatic ceiling "
                  "(policy 3.5). Escalate to a human agent instead.")
    else:
        return {}
    DENIALS.append(reason)
    print("   HOOK denied process_refund:", reason)
    return {"hookSpecificOutput": {
        "hookEventName": input_data["hook_event_name"],
        "permissionDecision": "deny",
        "permissionDecisionReason": reason,
    }}


HOOKS = {
    "PreToolUse": [HookMatcher(matcher="mcp__northwind__process_refund", hooks=[gate_and_ceiling])],
    "PostToolUse": [HookMatcher(matcher="mcp__northwind__get_customer", hooks=[record_verification])],
}

REFUND_LOG.clear(); VERIFIED.clear(); HANDOFFS.clear(); DENIALS.clear()
text, used, _ = await run(
    "alice@example.com wants ORD-999 refunded: it arrived damaged, it is inside the 30-day "
    "window, and the amount is 720.00. Verify her, then process the refund.",
    client_mode=True, hooks=HOOKS)

assert REFUND_LOG == [], f"the ceiling hook must stop the refund, log={REFUND_LOG}"
assert DENIALS, "the model never reached process_refund, so the hook was never exercised"
assert "ceiling" in DENIALS[-1], f"wrong rule fired: {DENIALS[-1]}"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__northwind__get_customer,mcp__northwind__lookup_order,mcp__northwind


   tool: mcp__northwind__get_customer {"email": "alice@example.com"}


   tool: mcp__northwind__lookup_order {"order_id": "ORD-999"}


   tool: mcp__northwind__read_policy {"section": "2.2"}


...

   tool: mcp__northwind__read_policy {"section": "2.4"}


   tool: mcp__northwind__process_refund {"order_id": "ORD-999", "amount": 720}
   HOOK denied process_refund: $720.00 is over the $500.00 automatic ceiling (policy 3.5). Escalate to a human agent instead.


   tool: mcp__northwind__read_policy {"section": "3.5"}


   tool: mcp__northwind__escalate_to_human {"customer_id": "CUST-001", "root_cause": "Alice Chen (CUST-001, Gold tier, alice@example.



RESULT: **Alice is verified, her claim is fully valid — but I could not process the refund. It's escalated as ticket ESC-9001.**

**Verification passed on every point:**
- Identity: alice@example.com → Alice Chen, CUST-001, Gold tier
- Ownership: ORD-999 belongs to CUST-001 — matches, not someone else's order
- Amount: $720.00 on record, matches her stated figure exactly
- Condition: recorded as `damaged`


### PostToolUse: one format, whatever the tool returned

One tool reports times as a Unix timestamp, another as ISO 8601. The hook attaches the normalised
form as `additionalContext`, so Claude reads one format regardless of which tool answered.

In [10]:
import datetime as dt
import re

@tool("shipment_unix", "Shipment events for an order, timestamped as Unix seconds.", {"order_id": str})
async def t_ship_unix(args):
    return ok({"order_id": args["order_id"], "shipped_at": 1790553600})  # 2026-09-28Z


@tool("shipment_iso", "Shipment events for an order, timestamped as ISO 8601.", {"order_id": str})
async def t_ship_iso(args):
    return ok({"order_id": args["order_id"], "shipped_at": "2026-09-28T00:00:00Z"})


shipping = create_sdk_mcp_server(name="shipping", version="1.0.0",
                                 tools=[t_ship_unix, t_ship_iso])

NORMALISED: list[str] = []


async def normalise_dates(input_data, tool_use_id, context):
    """PostToolUse: convert whatever the tool said into one ISO 8601 string."""
    blob = json.dumps(input_data.get("tool_response"))
    found = re.search(r"[^0-9](1[0-9]{9})[^0-9]", blob)   # a 10-digit Unix second
    if not found:
        return {}
    stamp = dt.datetime.fromtimestamp(int(found.group(1)), dt.UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
    NORMALISED.append(stamp)
    print("   HOOK normalised a Unix timestamp to", stamp)
    return {"hookSpecificOutput": {"hookEventName": "PostToolUse",
                                   "additionalContext": f"shipped_at (ISO 8601): {stamp}"}}


text, used, _ = await run(
    "Call shipment_unix and shipment_iso for ORD-123 and tell me both shipped dates "
    "in ISO 8601, one line each.",
    client_mode=True,
    mcp_servers={"shipping": shipping},
    allowed_tools=["mcp__shipping__*"],
    disallowed_tools=["Bash"],   # the hook does the conversion, not a shell
    hooks={"PostToolUse": [HookMatcher(matcher="mcp__shipping__shipment_unix",
                                       hooks=[normalise_dates])]})

assert NORMALISED == ["2026-09-28T00:00:00Z"], f"hook did not normalise, got {NORMALISED}"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__shipping__shipment_unix,mcp__shipping__shipment_iso", "max_results"


   tool: mcp__shipping__shipment_unix {"order_id": "ORD-123"}
   HOOK normalised a Unix timestamp to 2026-09-28T00:00:00Z


   tool: mcp__shipping__shipment_iso {"order_id": "ORD-123"}



RESULT: shipment_unix: 2026-09-28T00:00:00Z (converted from 1790553600)
shipment_iso: 2026-09-28T00:00:00Z

Both sources agree. I verified the Unix→ISO conversion independently: 1790553600 ÷ 86400 = 20724 exactly, so it lands on midnight UTC, and day 20724 after the epoch is day-of-year 271 in non-leap 2026 — September 28.


## Module 4 — Handling failure

`"Operation failed"` tells the agent nothing. Each error below carries a **category**
(transient / validation / permission / business), a **retryable** flag, and a description in words a
customer could understand. A timeout gets retried. A policy violation does not.

In [11]:
ATTEMPTS = {"flaky_lookup": 0}


@tool("flaky_lookup", "Look up an order in the warehouse system. May time out.", {"order_id": str})
async def t_flaky(args):
    ATTEMPTS["flaky_lookup"] += 1
    if ATTEMPTS["flaky_lookup"] == 1:
        return err("transient", True, "The warehouse service timed out after 5s. Safe to retry.")
    return ok({"order_id": args["order_id"], "location": "Bay 7"})


@tool("gift_card_refund", "Refund an order to a gift card.", {"order_id": str})
async def t_gift(args):
    return err("business", False,
               "Orders paid by card cannot be refunded to a gift card (policy 6.2). "
               "Refund to the original card instead.")


flaky = create_sdk_mcp_server(name="flaky", version="1.0.0", tools=[t_flaky, t_gift])

ATTEMPTS["flaky_lookup"] = 0
text, used, _ = await run(
    "Find where ORD-123 is, then try to refund it to a gift card. "
    "Retry anything retryable; do not retry anything that is not.",
    mcp_servers={"flaky": flaky}, allowed_tools=["mcp__flaky__*"])

assert ATTEMPTS["flaky_lookup"] >= 2, "a retryable timeout should have been retried"
assert used.count("mcp__flaky__gift_card_refund") == 1, \
    f"a business rule violation must not be retried, got {used}"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__flaky__flaky_lookup,mcp__flaky__gift_card_refund", "max_results": 5


   tool: mcp__flaky__flaky_lookup {"order_id": "ORD-123"}


   tool: mcp__flaky__flaky_lookup {"order_id": "ORD-123"}


   tool: mcp__flaky__gift_card_refund {"order_id": "ORD-123"}



RESULT: Done — here's where things landed:

**ORD-123 location: Bay 7**

**Lookup** — failed on the first call with a transient timeout (warehouse service timed out after 5s, explicitly marked retryable). I retried once and it succeeded.

**Gift card refund** — failed and I did **not** retry it. The error was a business-rule rejection, not a transient fault:

> Orders paid by card cannot be refunded to a 


### Empty is not failure

"The search ran and found nothing" is a success with zero rows. "The search could not run" is an
error. Report them differently or the agent draws the wrong conclusion.

In [12]:
@tool("search_orders", "Search orders by keyword. Returns matches, possibly none.", {"q": str})
async def t_search(args):
    matches = [k for k, v in ORDERS.items() if args["q"].lower() in json.dumps(v).lower()]
    return ok({"query": args["q"], "matches": matches, "count": len(matches)})


@tool("search_archive", "Search the archive. The archive is offline right now.", {"q": str})
async def t_archive(args):
    return err("transient", True, "The archive index is offline. Nothing was searched.")


searches = create_sdk_mcp_server(name="searches", version="1.0.0", tools=[t_search, t_archive])

text, used, _ = await run(
    "Search orders for 'damaged', then search the archive for 'damaged'. "
    "Say clearly for each one whether it found nothing or could not run. Do not retry.",
    mcp_servers={"searches": searches}, allowed_tools=["mcp__searches__*"])

low = text.lower()
assert "mcp__searches__search_orders" in used and "mcp__searches__search_archive" in used
assert "offline" in low or "could not" in low or "couldn't" in low, \
    "the agent must report the archive as a failed call, not as an empty result"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__searches__search_orders,mcp__searches__search_archive", "max_result


   tool: mcp__searches__search_orders {"q": "damaged"}


   tool: mcp__searches__search_archive {"q": "damaged"}



RESULT: Here's each one separately:

**Orders search — ran successfully, found matches.** Not an empty result: 2 matches, `ORD-123` and `ORD-999`.

**Archive search — could not run.** This is *not* "found nothing." The archive index is offline, and the error explicitly states nothing was searched. There may or may not be matching archive records — we simply have no information either way. The error is mar


## Module 5 — Breaking down complex tasks

**Fixed steps (prompt chaining)** when you know the steps: review each file on its own, then one
final pass over how they connect. Attention never spreads thin across all the files at once.

In [13]:
FILES = {
    "loop.py":    "while True:\n    r = call_model(msgs)\n    if 'done' in r.text: break",
    "tools.py":   "def process_refund(order_id, amount):\n    return charge_back(order_id, amount)",
    "verify.py":  "def get_customer(email):\n    return DB.get(email)  # may return None",
}


def ask(prompt: str, max_tokens: int = 400) -> str:
    resp = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                  messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in resp.content if b.type == "text")


# Step 1 — one call per file, in isolation.
per_file = {name: ask(f"In one sentence, name the single biggest risk in this file.\n\n"
                      f"# {name}\n{body}") for name, body in FILES.items()}
for name, note in per_file.items():
    print(f"{name:>10}: {note.strip()[:110]}")

# Step 2 — one final call that only sees the summaries, and looks across them.
joined = "\n".join(f"{n}: {t.strip()}" for n, t in per_file.items())
connections = ask("These are per-file notes from one codebase. In two sentences, name the risk that "
                  "only appears when the files are used together.\n\n" + joined)
print("\nacross files:", connections.strip())

assert len(per_file) == len(FILES), "one call per file"
assert connections.strip(), "the final pass produced nothing"

   loop.py: The biggest risk is an **infinite loop** — if the model never returns a response containing `'done'`, the loop
  tools.py: The biggest risk is that `charge_back` is called but never defined or imported in this file, meaning any refun
 verify.py: The biggest risk is that `get_customer` can return `None` but there is no null check shown, meaning any code t



across files: When `loop.py` drives repeated calls that eventually invoke `verify.py`'s `get_customer`, a `None` return will cause a crash — but because the loop in `loop.py` has no termination guard beyond a `'done'` signal, that crash may be silently swallowed or the loop may retry endlessly before `tools.py`'s `charge_back` `NameError` is ever even reached. The compounded risk is that a single bad customer lookup can trap the system in an unresolvable error-retry cycle, burning API costs indefinitely while no refund is ever successfully processed.


**Multi-part requests**: a customer asks about three things at once. Split, look into each, then
combine into one answer — otherwise the agent answers the loudest concern and drops the rest.

In [14]:
split = ask("Split this support message into separate concerns. Return ONLY a JSON array of "
            "objects with keys \"id\", \"order_id\", \"ask\". No prose.\n\n" + ALICE_TICKET)
concerns = json.loads(split[split.index("["):split.rindex("]") + 1])
for c in concerns:
    print(c)

assert len(concerns) >= 3, f"Alice raised at least three concerns, got {len(concerns)}"
assert {c.get("order_id") for c in concerns} >= {"ORD-123", "ORD-456", "ORD-789"}

{'id': 'alice@example.com', 'order_id': 'ORD-123', 'ask': 'Order arrived damaged and customer wants a refund.'}
{'id': 'alice@example.com', 'order_id': 'ORD-456', 'ask': 'Card was charged twice for this order.'}
{'id': 'alice@example.com', 'order_id': 'ORD-789', 'ask': 'Shoes purchased 40 days ago — customer asking if a return is still possible.'}
{'id': 'alice@example.com', 'order_id': None, 'ask': 'Customer threatens to file a chargeback if issues are not resolved today.'}


## Module 6 — Going multi-agent

A coordinator plus three specialists. Subagents start with a blank memory: they never see the
coordinator's conversation, so everything they need goes in their prompt. Several `Task` calls in one
response run in parallel.

In [15]:
SOURCE_CONTRACT = (
    "\n\nReturn a JSON array. Each element has \"claim\", \"evidence\" (the exact value "
    "or clause you read) and \"source\" (the tool call or policy section it came from)."
)

AGENTS = {
    "search": AgentDefinition(
        description="Looks up orders and customers. Use for the state of a specific record.",
        prompt="You look up records and report what the tools returned." + SOURCE_CONTRACT,
        tools=["mcp__northwind__lookup_order", "mcp__northwind__get_customer"],
        model="sonnet",
    ),
    "analysis": AgentDefinition(
        description="Answers questions that depend on written policy, such as return windows.",
        prompt="You read policy and decide what it allows. Always cite the section number."
               + SOURCE_CONTRACT,
        tools=["mcp__northwind__read_policy"],
        model="sonnet",
    ),
}

started = time.time()
text, used, _ = await run(
    f"Today is {TODAY}. In ONE response spawn both subagents in parallel: "
    "'search' to look up ORD-789, and 'analysis' to read policy 2.2. "
    "Paste what each needs into its prompt — they cannot see this conversation. "
    "Then say in one line whether ORD-789 is still returnable.",
    agents=AGENTS, max_turns=10)
print(f"\nwall clock: {time.time() - started:.1f}s")

spawns = [u for u in used if u in ("Agent", "Task")]
assert len(spawns) >= 2, f"expected two subagent spawns, got {used}"
assert "ORD-789" in text, "the coordinator lost the order it was asked about"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: search | Look up order ID "ORD-789" using the lookup_order tool.

Report back A


   tool: mcp__northwind__lookup_order {"order_id": "ORD-789"}


spawned: analysis | Read policy section "2.2" using the read_policy tool.

Report back the


   tool: mcp__northwind__read_policy {"section": "2.2"}



RESULT: No — ORD-789 was delivered 2026-09-06, which is 40 days ago, past policy 2.2's 30-day-from-delivery window (day 31+ is refused).

wall clock: 20.3s


## Module 7 — Context and state

Long tasks fill the context and quality drops. Two of the fixes, run here: a **scratchpad file** the
agent writes findings to and reads back, and **sessions** — `resume` to continue, `fork_session` to
try two approaches from the same starting point.

In [16]:
SCRATCH = Path("scratchpad.md")
SCRATCH.unlink(missing_ok=True)


@tool("write_note", "Append one finding to the shared scratchpad file.", {"line": str})
async def t_write_note(args):
    with SCRATCH.open("a") as fh:
        fh.write(args["line"].rstrip() + "\n")
    return ok({"written": args["line"]})


@tool("read_notes", "Read every finding written to the scratchpad so far.", {})
async def t_read_notes(args):
    return ok({"notes": SCRATCH.read_text() if SCRATCH.exists() else ""})


scratch = create_sdk_mcp_server(name="scratch", version="1.0.0",
                                tools=[t_write_note, t_read_notes])

await run("Look up ORD-123 and ORD-456, and after each lookup write one short finding "
          "to the scratchpad with write_note. Do not summarise at the end.",
          mcp_servers={"northwind": northwind, "scratch": scratch},
          allowed_tools=["mcp__northwind__*", "mcp__scratch__*"])

print("\n--- scratchpad.md ---")
print(SCRATCH.read_text())

# A second agent, with no memory of the first, reads the findings back.
text, used, _ = await run("Read the scratchpad and tell me which order was damaged.",
                          mcp_servers={"scratch": scratch}, allowed_tools=["mcp__scratch__*"])

assert SCRATCH.exists() and SCRATCH.read_text().strip(), "nothing was written to the scratchpad"
assert "ORD-123" in text, "the second agent could not recover the finding from the file"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__northwind__lookup_order,mcp__scratch__write_note", "max_results": 5


   tool: mcp__northwind__lookup_order {"order_id": "ORD-123"}


   tool: mcp__northwind__lookup_order {"order_id": "ORD-456"}


   tool: mcp__scratch__write_note {"line": "ORD-123 (CUST-001, $149.99): delivered 2026-09-28, condition damaged, charged on


...


RESULT: Both lookups are done and both findings are written to the scratchpad.

--- scratchpad.md ---
ORD-123 (CUST-001, $149.99): delivered 2026-09-28, condition damaged, charged once — damage claim, refund candidate.
ORD-456 (CUST-001, $89.00): delivered 2026-09-22 in ok condition but charged 2x — duplicate charge, one $89.00 refund owed.



⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__scratch__read_notes", "max_results": 5}


   tool: mcp__scratch__read_notes {}



RESULT: **ORD-123** was the damaged order.

- **ORD-123** (CUST-001, $149.99) — delivered 2026-09-28, condition **damaged**, charged once → damage claim, refund candidate.
- ORD-456 (CUST-001, $89.00) — delivered 2026-09-22 in ok condition, but charged twice → separate issue, one $89.00 refund owed for the duplicate charge.


### Resume and fork

One session establishes a fact. `resume` continues it. `fork_session=True` branches it, so two
approaches start from the same state without one overwriting the other.

In [17]:
base_opts = dict(mcp_servers={"northwind": northwind},
                 allowed_tools=["mcp__northwind__*"],
                 setting_sources=[], permission_mode="bypassPermissions", max_turns=6)


async def one_shot(prompt, **overrides):
    opts = {**base_opts, **overrides}
    out, sid = "", None
    async with ClaudeSDKClient(options=ClaudeAgentOptions(**opts)) as c:
        await c.query(prompt)
        async for m in c.receive_response():
            if isinstance(m, ResultMessage):
                sid = m.session_id
                out = m.result or ""
    return out, sid


first, sid = await one_shot("Look up ORD-789 and remember its amount. Reply with just the amount.")
print("session", sid, "->", first.strip()[:80])

ASK = ("{q} Answer in one line, and start it with the exact amount you remember "
       "for ORD-789 — do not look it up again.")
branch_a, sid_a = await one_shot(ASK.format(q="Should we refund that amount in full?"),
                                 resume=sid, fork_session=True)
branch_b, sid_b = await one_shot(ASK.format(q="Should we offer store credit instead?"),
                                 resume=sid, fork_session=True)
print("\nbranch A:", branch_a.strip()[:160])
print("branch B:", branch_b.strip()[:160])

assert sid_a != sid and sid_b != sid, "fork_session must start new sessions"
assert sid_a != sid_b, "the two branches must be independent"
assert "812" in branch_a and "812" in branch_b, "a forked branch lost the inherited context"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


session ed76995b-ef29-47f5-a921-2da002d85241 -> $812.40


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors



branch A: $812.40 — I wouldn't refund it without more justification: the order shows delivered, condition "ok", and only one charge, so there's no overcharge or defect on
branch B: $812.40 — store credit could work as an option, but since the order was delivered in good condition and charged only once, there's no refund trigger here; offer


## Module 8 — Escalation

The human picking this up cannot see the conversation. Four fields: customer ID, root cause, amount,
recommended next action.

In [18]:
HANDOFFS.clear(); REFUND_LOG.clear(); VERIFIED.clear()

text, used, _ = await run(
    "alice@example.com wants the full 812.40 on ORD-789 back. Verify her, check policy 3.5, "
    "and if the automated system may not do it, escalate to a human with everything they need.",
    client_mode=True, hooks=HOOKS)

print("\nhandoff:", json.dumps(HANDOFFS, indent=2)[:600])

assert HANDOFFS, "nothing was escalated"
h = HANDOFFS[-1]
for field in ("customer_id", "root_cause", "amount", "recommended_action"):
    assert str(h.get(field, "")).strip(), f"the handoff is missing {field}"
assert REFUND_LOG == [], "the ceiling still holds"

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__northwind__get_customer,mcp__northwind__lookup_order,mcp__northwind


   tool: mcp__northwind__get_customer {"email": "alice@example.com"}


   tool: mcp__northwind__lookup_order {"order_id": "ORD-789"}


   tool: mcp__northwind__read_policy {"section": "3.5"}


   tool: mcp__northwind__escalate_to_human {"customer_id": "CUST-001", "amount": 812.4, "root_cause": "Customer Alice Chen (CUST-001,



RESULT: Escalated as **ticket ESC-9001**. No refund was processed.

**Verification**
- `alice@example.com` → Alice Chen, CUST-001, Gold tier.
- ORD-789 belongs to CUST-001 — the order is hers, so the request is legitimate on its face.
- Order: $812.40, delivered 2026-09-06, condition `ok`, charged once.

**Policy 3.5** — "Refunds above $500 require a human agent. The automated system must escalate." At $8

handoff: [
  {
    "customer_id": "CUST-001",
    "amount": 812.4,
    "root_cause": "Customer Alice Chen (CUST-001, Gold tier, alice@example.com) requests a full refund of $812.40 on ORD-789. Identity verified via get_customer; ORD-789 is confirmed to belong to CUST-001. Order facts: amount $812.40, status delivered, delivered_on 2026-09-06, condition \"ok\", charges 1. Automated refund is blocked by policy 3.5: \"Refunds above $500 require a human agent. The automated system must escalate.\" $812.40 exceeds the $500 automated ceiling, so this is a threshold escalation, not a fault

## Capstone — all of it, on one message with four problems

Four tools, both hooks, structured errors, subagents in parallel, and an escalation. Alice's refunds:
ORD-123 ($149.99, damaged) and ORD-456 ($89.00, double charge) are under the ceiling; ORD-789 is over
it, and day 40 is outside the 30-day window.

In [19]:
REFUND_LOG.clear(); HANDOFFS.clear(); VERIFIED.clear()

text, used, _ = await run(
    "Northwind support. The customer wrote:\n\n" + ALICE_TICKET +
    f"\n\nToday is {TODAY}. Verify her first. Check policy before any refund. "
    "Refund what you may refund, escalate what you may not, and answer every concern.",
    client_mode=True, hooks=HOOKS, max_turns=16)

print("refunds :", REFUND_LOG)
print("handoffs:", [h.get("root_cause", "")[:60] for h in HANDOFFS])

assert all(r["amount"] <= REFUND_CEILING for r in REFUND_LOG), \
    f"the ceiling hook must hold on the capstone run: {REFUND_LOG}"
for order in ("ORD-123", "ORD-456", "ORD-789"):
    assert order in text, f"the final answer dropped {order}"
print("\nCAPSTONE OK")

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {"query": "select:mcp__northwind__get_customer,mcp__northwind__lookup_order,mcp__northwind


   tool: mcp__northwind__get_customer {"email": "alice@example.com"}


   tool: mcp__northwind__lookup_order {"order_id": "ORD-123"}


   tool: mcp__northwind__lookup_order {"order_id": "ORD-456"}


...

   tool: mcp__northwind__read_policy {"section": "4.6"}


   tool: mcp__northwind__process_refund {"order_id": "ORD-123", "amount": 149.99}


   tool: mcp__northwind__process_refund {"order_id": "ORD-456", "amount": 89}


   tool: mcp__northwind__escalate_to_human {"customer_id": "CUST-001", "root_cause": "ORD-789 (shoes, $812.40, delivered 2026-09-06) 



RESULT: Case handled. Identity verified first, policy read before any money moved, and all three orders confirmed to belong to CUST-001.

**Actions taken**

- **ORD-123 — refunded $149.99.** Delivered 2026-09-28, day 18, inside the 30-day window (2.2), arrived damaged, under the $500 cap. Clean approval.
- **ORD-456 — refunded $89.00.** `charges: 2` confirmed the double billing; 4.1 refunds duplicates in 
refunds : [{'order_id': 'ORD-123', 'amount': 149.99}, {'order_id': 'ORD-456', 'amount': 89}]
handoffs: ['ORD-789 (shoes, $812.40, delivered 2026-09-06) is day 40 pos']

CAPSTONE OK
